# 🎬 VSL Forge — MECProAI
## Geração de Vídeos dos Cursos + Criativos de Campanha

### ⚠️ ANTES DE RODAR:
1. Vá em **Runtime → Change runtime type → T4 GPU**
2. Execute as células **em ordem**
3. Na Célula 3, faça upload dos arquivos: `vsl_forge_v2.py`, `vsl_service.py`, `academy_cursos.json`

### ⏱️ Tempo estimado:
- Modo `--draft` (teste): ~2-3 min por vídeo
- Modo final (qualidade): ~5-10 min por vídeo
- 8 cursos completos: ~40-80 min no total

## 📦 CÉLULA 1 — Instalar dependências

In [ ]:
import subprocess, sys

# Sistema
subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True)

# Python
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi', 'uvicorn', 'python-multipart', 'aiofiles',
    'pyngrok', 'nest-asyncio', 'requests', 'Pillow',
    'tqdm', 'elevenlabs', 'huggingface_hub'
], capture_output=True)

# Verificar FFmpeg
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
ffmpeg_ok = result.returncode == 0
print('✅ FFmpeg:', 'OK' if ffmpeg_ok else '❌ FALHOU')
print('✅ Dependências Python: OK')
print('\n🟢 Pronto para a próxima célula!')

## 🔑 CÉLULA 2 — Configurar API Keys (já preenchidas)

In [ ]:
import os

# ══════════════════════════════════════════════
# KEYS JÁ CONFIGURADAS — não precisa alterar
# ══════════════════════════════════════════════

ELEVENLABS_API_KEY  = "sk_d524c8cb9bdbe0e620dea85ff383eae12ce9eba2d7ab8a3f"
HUGGINGFACE_API_KEY = "hf_SUA_KEY_AQUI"
NGROK_AUTH_TOKEN    = "3Bi8Slx2ypO1ep9gDaUvdFfwO5h_2bikKxPRea46yeGetHzV9"
VSL_SECRET_KEY      = "mecpro-vsl-2026"


os.environ['ELEVENLABS_API_KEY']  = ELEVENLABS_API_KEY
os.environ['HUGGINGFACE_API_KEY'] = HUGGINGFACE_API_KEY
os.environ['VSL_SECRET_KEY']      = VSL_SECRET_KEY
os.environ['OUTPUT_DIR']          = '/content/vsl_outputs'

os.makedirs('/content/vsl_outputs', exist_ok=True)
os.makedirs('/content/academy_output', exist_ok=True)

print('✅ ElevenLabs  :', ELEVENLABS_API_KEY[:12] + '...')
print('✅ HuggingFace :', HUGGINGFACE_API_KEY[:12] + '...')
print('✅ VSL Key     :', VSL_SECRET_KEY)
print('✅ ngrok        :', NGROK_AUTH_TOKEN[:12] + '...')

## 📥 CÉLULA 3 — Upload dos arquivos

In [ ]:
from google.colab import files
import os

print('📤 Faça upload dos 3 arquivos:')
print('   1. vsl_forge_v2.py')
print('   2. vsl_service.py')
print('   3. academy_cursos.json')
print('   4. split_cursos.py  (opcional)')
print()

uploaded = files.upload()

for filename, data in uploaded.items():
    dest = f'/content/{filename}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  ✅ {filename} ({len(data)//1024} KB)')

print()
needed = ['vsl_forge_v2.py', 'vsl_service.py', 'academy_cursos.json']
all_ok = True
for f in needed:
    exists = os.path.exists(f'/content/{f}')
    print(f'  {"✅" if exists else "❌ FALTANDO"} {f}')
    if not exists: all_ok = False

print()
if all_ok:
    print('🟢 Tudo pronto! Pode seguir para a próxima célula.')
else:
    print('🔴 Faça upload dos arquivos que estão faltando antes de continuar.')

## 📚 CÉLULA 4 — Listar cursos disponíveis

In [ ]:
import json

with open('/content/academy_cursos.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

cursos = data['cursos']
print(f'📚 {len(cursos)} cursos encontrados:\n')
total_cenas = 0
for i, c in enumerate(cursos, 1):
    n = len(c['scenes'])
    total_cenas += n
    print(f'  {i}. {c["slug"]:45s} ({n} cenas)')

print(f'\n  Total de cenas: {total_cenas}')
print(f'  Tempo estimado (qualidade final): ~{total_cenas * 7} min')
print(f'  Tempo estimado (modo draft):      ~{total_cenas * 2} min')

## 🎬 CÉLULA 5 — Gerar vídeos dos cursos
### Opções:
- `MODO_DRAFT = True` → rápido, 480p, para testar
- `MODO_DRAFT = False` → qualidade final, 1080p
- `CURSO_ESPECIFICO = None` → gera todos os cursos
- `CURSO_ESPECIFICO = 'campanha-zero-mecpro'` → gera só um

In [ ]:
import json, subprocess, sys, os
from pathlib import Path

# ══════════════════════════════════════════════
# CONFIGURAÇÃO — altere aqui
MODO_DRAFT       = True    # True = rápido/teste | False = qualidade final
CURSO_ESPECIFICO = None    # None = todos | ex: 'campanha-zero-mecpro'
# ══════════════════════════════════════════════

# Carregar cursos
with open('/content/academy_cursos.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

cursos = data['cursos']
if CURSO_ESPECIFICO:
    cursos = [c for c in cursos if c['slug'] == CURSO_ESPECIFICO]
    if not cursos:
        print(f'❌ Curso "{CURSO_ESPECIFICO}" não encontrado!')
        print('Disponíveis:', [c['slug'] for c in data['cursos']])
        raise SystemExit(1)

# Salvar JSONs individuais
output_dir = Path('/content/academy_output')
output_dir.mkdir(exist_ok=True)
for c in cursos:
    path = output_dir / f"{c['slug']}.json"
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(c, f, indent=2, ensure_ascii=False)

modo_str = '⚡ DRAFT (480p, rápido)' if MODO_DRAFT else '🎬 FINAL (1080p, qualidade)'
print(f'Modo: {modo_str}')
print(f'Cursos para gerar: {len(cursos)}\n')

resultados = []
for i, curso in enumerate(cursos, 1):
    slug = curso['slug']
    config = f'/content/academy_output/{slug}.json'
    print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    print(f'[{i}/{len(cursos)}] 🎬 {slug}')
    print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

    cmd = [sys.executable, '/content/vsl_forge_v2.py',
           '--config', config, '--resume']
    if MODO_DRAFT:
        cmd.append('--draft')

    result = subprocess.run(cmd, text=True)
    ok = result.returncode == 0
    resultados.append((slug, ok))
    print(f'  {"✅ OK" if ok else "❌ FALHOU"}: {slug}\n')

print('\n' + '═'*50)
print('RESUMO FINAL')
print('═'*50)
for slug, ok in resultados:
    print(f'  {"✅" if ok else "❌"} {slug}')

sucessos = sum(1 for _, ok in resultados if ok)
print(f'\n  {sucessos}/{len(resultados)} cursos gerados com sucesso')
print(f'\n📁 Vídeos em: /content/vsl_build/')

## 💾 CÉLULA 6 — Salvar vídeos no Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, os

# Montar Google Drive
drive.mount('/content/drive')

# Pasta de destino no Drive
DRIVE_DEST = Path('/content/drive/MyDrive/MECProAI/VSL_Videos')
DRIVE_DEST.mkdir(parents=True, exist_ok=True)

# Copiar todos os MP4 finais
build_dir = Path('/content/vsl_build')
copiados = 0

if not build_dir.exists():
    print('⚠️  Nenhum vídeo gerado ainda. Rode a Célula 5 primeiro.')
else:
    for mp4 in build_dir.rglob('final/*.mp4'):
        dest = DRIVE_DEST / mp4.name
        shutil.copy2(mp4, dest)
        print(f'  ✅ {mp4.name} → Drive')
        copiados += 1

    print(f'\n🎉 {copiados} vídeo(s) salvos em:')
    print(f'   📁 MECProAI/VSL_Videos no seu Google Drive')

## 🌐 CÉLULA 7 — Iniciar API + ngrok (opcional)
**Só precisa desta célula se quiser que o MECProAI chame a API automaticamente.**  
Para gerar os vídeos manualmente, as células 5 e 6 já são suficientes.

In [ ]:
import nest_asyncio, uvicorn, threading, time, sys, importlib.util
from pyngrok import ngrok, conf

nest_asyncio.apply()

conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Carregar o vsl_service
sys.path.insert(0, '/content')
spec = importlib.util.spec_from_file_location('vsl_service', '/content/vsl_service.py')
vsl_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vsl_module)
app = vsl_module.app

# Iniciar servidor em thread separada
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)

# Túnel ngrok com domínio FIXO (não muda ao reiniciar)
NGROK_DOMAIN = 'arcane-embryologic-tiffiny.ngrok-free.dev'
tunnel = ngrok.connect(8000, 'http', domain=NGROK_DOMAIN)
public_url = f'https://{NGROK_DOMAIN}'

print('=' * 60)
print('🚀 VSL Service online!')
print('=' * 60)
print(f'\n🌐 URL pública: {public_url}')
print(f'\n📋 Cole no Render (Environment Variables):')
print(f'   VSL_SERVICE_URL = {public_url}')
print(f'   VSL_SECRET_KEY  = {VSL_SECRET_KEY}')
print()
print('✅ Domínio FIXO — essa URL nunca muda!')
print('   Salve no Render uma única vez e esqueça.')

## 🔄 CÉLULA 8 — Manter Colab vivo (anti-timeout)
Execute enquanto gera vídeos para o Colab não desconectar.

In [ ]:
import time
from pathlib import Path

print('🔄 Anti-timeout ativo. Ctrl+C ou ■ para parar.\n')

minutos = 0
while True:
    time.sleep(60)
    minutos += 1

    # Conta vídeos já gerados
    videos = list(Path('/content/vsl_build').rglob('final/*.mp4')) if Path('/content/vsl_build').exists() else []
    print(f'  ⏱️  {minutos} min — {len(videos)} vídeo(s) gerado(s) até agora', flush=True)